# Module 3: Operating and Optimizing Apache Iceberg

This notebook contains code examples for Module 3 videos.

⚠️ **IMPORTANT**: Before running cells in this notebook, please complete the following:
1. **Run the Setup notebook first**: Open and run `Videos/Setup.ipynb` to download the required data
2. **Shut down other running kernels**: Go to the **Running Terminals and Kernels** section (left sidebar) and shut down any other active notebooks to avoid memory issues

## Setup

Initialize Spark session for Module 3 examples.

## Video: Ingesting Data into Apache Iceberg

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module3_Operating_and_Optimizing") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

### Basic Write Operations

INSERT, DELETE, and OVERWRITE examples

In [ ]:
# Setup: Create trips_by_day table with a few days of data for demonstration
spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_by_day")

spark.sql("""
    CREATE TABLE polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-09.parquet`
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
""")

In [ ]:
# Insert operation - adds new rows to the table
# This is purely additive and the fastest write operation

spark.sql("""
    INSERT INTO polaris.taxi.trips_by_day
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-09.parquet`
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
""")

In [ ]:
# Delete operation with partition filter
# When aligned with partition boundaries, this is a metadata-only operation

spark.sql("""
    DELETE FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-09-01'
      AND tpep_pickup_datetime < '2023-09-04'
""")

## Video: Copy on Write and Merge on Read

In [ ]:
# Overwrite operation - atomic delete + insert
# Replaces data matching a predicate with new data

spark.sql("""
    INSERT OVERWRITE polaris.taxi.trips_by_day
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
""")

In [ ]:
# Copy-on-Write (COW) mode update
# Rewrites entire affected files with modifications applied
# Slower writes, faster reads

# Set table to use copy-on-write mode for updates
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    SET TBLPROPERTIES (
        'write.update.mode' = 'copy-on-write'
    )
""")

spark.sql("""
    UPDATE polaris.taxi.trips_by_day
    SET fare_amount = fare_amount * -1
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
""")

In [ ]:
# View the result of COW update
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        PULocationID,
        fare_amount
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
      AND PULocationID = 132
    LIMIT 10
""").show()

In [ ]:
# Merge-on-Read (MOR) mode update
# Writes delete files and small data files with changes
# Faster writes, slower reads

# Set table to use merge-on-read mode for updates
spark.sql("""
    ALTER TABLE polaris.taxi.trips_by_day
    SET TBLPROPERTIES (
        'write.update.mode' = 'merge-on-read'
    )
""")

spark.sql("""
    UPDATE polaris.taxi.trips_by_day
    SET tip_amount = tip_amount * 100
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
""")

In [ ]:
# View the result of MOR update
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        PULocationID,
        tip_amount
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-09-10'
      AND tpep_pickup_datetime < '2023-09-13'
    LIMIT 10
""").show()

## Video: Handling Concurrency in Apache Iceberg

In [ ]:
# View delete files in metadata
# MOR creates delete files that track which rows to ignore

spark.sql("""
    SELECT 
        SUBSTRING(file_path, INSTR(file_path, '/data/') + 1) as file_path,
        record_count,
        SUBSTRING(referenced_data_file, INSTR(referenced_data_file, '/data/') + 1) as referenced_data_file
    FROM polaris.taxi.trips_by_day.delete_files
    LIMIT 10
""").show(truncate=False)

## Video: Table Maintenance for Iceberg - The Basics

In [ ]:
# Setup: Create trips_by_month_maintenance table with multiple snapshots
# This table will be used to demonstrate various maintenance operations

spark.sql("DROP TABLE IF EXISTS polaris.taxi.trips_by_month_maintenance")

# Create table with first month's data
spark.sql("""
    CREATE TABLE polaris.taxi.trips_by_month_maintenance
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-07-01'
""")

print("CTAS Complete")

# Insert additional months to create more snapshots
for month in ['07', '08', '09', '10']:
    spark.sql(f"""
        INSERT INTO polaris.taxi.trips_by_month_maintenance
        SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-{month}.parquet`
        WHERE tpep_pickup_datetime >= '2023-{month}-01'
          AND tpep_pickup_datetime < '2023-{month}-01'::date + INTERVAL '1' MONTH
    """)
    print(f"Month-{month} Inserted")

In [ ]:
# View all snapshots - should show 5 snapshots (1 CTAS + 4 INSERT operations)
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation,
        summary['added-records'] as added_records
    FROM polaris.taxi.trips_by_month_maintenance.snapshots
    ORDER BY committed_at ASC
""").show(truncate=False)

In [ ]:
# Expire old snapshots to prevent metadata.json bloat
# This removes snapshots older than now, keeping only the last 2 snapshots

spark.sql("""
    CALL polaris.system.expire_snapshots(
        table => 'polaris.taxi.trips_by_month_maintenance',
        older_than => current_timestamp(),
        retain_last => 2
    )
""").show()

In [ ]:
# View remaining snapshots after expiration - should only show last 2 snapshots
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_by_month_maintenance.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

## Video: Table Maintenance for Iceberg - Compaction and Abandoned File Cleanup

In [ ]:
# View manifests before compaction
# Multiple small manifest files slow down query planning
spark.sql("""
    SELECT 
        SUBSTRING(path, INSTR(path, '/metadata/') + 1) as path,
        length,
        added_data_files_count,
        existing_data_files_count
    FROM polaris.taxi.trips_by_month_maintenance.manifests
""").show(truncate=False)

In [ ]:
# Rewrite manifests to consolidate small manifest files
# This speeds up query planning by reducing metadata overhead

spark.sql("""
    CALL polaris.system.rewrite_manifests(
        table => 'polaris.taxi.trips_by_month_maintenance'
    )
""").show()

In [ ]:
# View manifests after compaction
# Should show fewer, larger manifest files
spark.sql("""
    SELECT 
        SUBSTRING(path, INSTR(path, '/metadata/') + 1) as path,
        length,
        added_data_files_count,
        existing_data_files_count
    FROM polaris.taxi.trips_by_month_maintenance.manifests
""").show(truncate=False)

In [ ]:
# Rewrite Small Files so Example is Clearer

spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'polaris.taxi.trips_by_month_maintenance',
        strategy => 'binpack',
        options => map('target-file-size-bytes', '32777216')
    )
""").show()

In [ ]:
# Perform several small writes to create many small files
# Each insert adds just one day's worth of data from July 2023

for day in range(1, 8):  # Days 1-7
    day_str = f"{day:02d}"
    next_day = day + 1
    next_day_str = f"{next_day:02d}"
    
    spark.sql(f"""
        INSERT INTO polaris.taxi.trips_by_month_maintenance
        SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-07.parquet`
        WHERE tpep_pickup_datetime >= '2023-07-{day_str}'
          AND tpep_pickup_datetime < '2023-07-{next_day_str}'
    """)
    
print("Completed 7 small daily inserts")

In [ ]:
# View small files before compaction
# Files smaller than 8MB indicate suboptimal file sizes
spark.sql("""
    SELECT 
        COUNT(*) as small_file_count,
        SUM(file_size_in_bytes) / 1024 / 1024 as total_size_mb
    FROM polaris.taxi.trips_by_month_maintenance.files
    WHERE file_size_in_bytes < 8 * 1024 * 1024
""").show()

# Show distribution of file sizes by partition before compaction
spark.sql("""
    SELECT 
        partition,
        COUNT(*) as file_count,
        MIN(file_size_in_bytes / 1024 / 1024) as min_size_mb,
        AVG(file_size_in_bytes / 1024 / 1024) as avg_size_mb,
        MAX(file_size_in_bytes / 1024 / 1024) as max_size_mb,
        SUM(record_count) as total_records
    FROM polaris.taxi.trips_by_month_maintenance.files
    WHERE partition.tpep_pickup_datetime_month = 642
    GROUP BY partition

""").show(truncate=False)

In [ ]:
# Rewrite data files to compact small files into optimally-sized files
# Target file size: 32MB to consolidate our small daily files
# min-input-files: 1 means compact partitions with 2 or more files

spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'polaris.taxi.trips_by_month_maintenance',
        strategy => 'binpack',
        options => map('target-file-size-bytes', '32777216')
    )
""").show()

## Video: Writing efficiently to Iceberg Tables

In [ ]:
# Verify no small files remain after compaction
spark.sql("""
    SELECT 
        COUNT(*) as small_file_count
    FROM polaris.taxi.trips_by_month_maintenance.files
    WHERE file_size_in_bytes < 8 * 1024 * 1024
""").show()

# Show distribution of file sizes by partition after compaction
spark.sql("""
    SELECT 
        partition,
        COUNT(*) as file_count,
        MIN(file_size_in_bytes / 1024 / 1024) as min_size_mb,
        AVG(file_size_in_bytes / 1024 / 1024) as avg_size_mb,
        MAX(file_size_in_bytes / 1024 / 1024) as max_size_mb,
        SUM(record_count) as total_records
    FROM polaris.taxi.trips_by_month_maintenance.files
    WHERE partition.tpep_pickup_datetime_month = 642
    GROUP BY partition
""").show(truncate=False)

In [ ]:
# Remove orphan files - only run when necessary
# This is expensive and should be used sparingly

spark.sql("""
    CALL polaris.system.remove_orphan_files(
        table => 'polaris.taxi.trips_by_day',
        older_than => TIMESTAMP '2023-09-01 00:00:00'
    )
""").show()

## Video: Sort Orders

In [ ]:
# spark.sql("""
#    ALTER TABLE polaris.taxi.trips_sorted_example
#    WRITE ORDERED BY user_id, ts
# """)

### Partition Granularity Examples

Comparing different partition strategies (monthly vs. daily)

In [ ]:
# Define a sort order on a table
# Sort orders organize data within files according to specified columns
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_sorted_example """)

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_sorted_example
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.taxi.trips_sorted_example
    WRITE ORDERED BY PULocationID, tpep_pickup_datetime
""")

In [ ]:
# Insert data into the sorted table
# Data will be sorted by PULocationID and timestamp before writing

spark.sql("""
    INSERT INTO polaris.taxi.trips_sorted_example
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

In [ ]:
# View min/max bounds for sorted columns
# Sort order creates better min/max statistics for file skipping

spark.sql("""
    SELECT 
        partition,
        record_count,
        readable_metrics.PULocationID.lower_bound as PULocationID_min,
        readable_metrics.PULocationID.upper_bound as PULocationID_max
    FROM polaris.taxi.trips_sorted_example.files
    ORDER BY partition, PULocationID_min
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Example: Partition by month (low cardinality)
# Fewer partitions = larger files, fewer to open

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month_example
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# View partition statistics for monthly partitioning
spark.sql("""
    SELECT 
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(AVG(data_file.file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_month_example.entries
""").show()

In [ ]:
# Example: Partition by day (high cardinality)
# More partitions = more files, smaller file sizes

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day_example
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

## Video: Summary of Module 3

In [ ]:
# View partition statistics for daily partitioning
spark.sql("""
    SELECT 
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(AVG(data_file.file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_day_example.entries
    LIMIT 10
""").show()

In [ ]:
# Compare overall file statistics
spark.sql("""
    SELECT 
        'Monthly' as partition_type,
        COUNT(*) as total_files,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_month_example.files
    UNION ALL
    SELECT 
        'Daily' as partition_type,
        COUNT(*) as total_files,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_day_example.files
""").show()